# 2026-09-02_3 线1：pathway → module 生物学功能划分

对 U1 focal pathway（Macular 112 / Peripheral 244）按**生物学功能**划分 module。

- **全 celltype、全方向共用同一份词典**（module 空间只定义一次 → 三组与六 celltype 可比）；
- 词典为**有序规则**：先匹配到的规则生效（特异规则在前，通用规则在后）——
  规则可复现、可写进 methods、可单条修改后全表重算；
- module 数由 pathway 实际涉及的生物学类目决定，不预设 k。

**共享 module 空间**：词典只应用一次、作用在双方向 pathway 的并集上，再按方向拆表 →
同名 pathway 两方向归属必然一致，27 个 module 为共享空间（各方向只是用到其中一部分，
这是生物学结果而非方法差异）。

输出两个两列表格（pathway, module）：Macular 与 Peripheral 各一份；
另出合并长表与 27-module 全空间对照表，便于后续跨方向比较。

> **注意（2026-09-03）**：本脚本的输出为**词典法草稿**（文件名带 `_draft`）。
> 正式使用的 module 映射是作者人工审定后的 `2026-09-02_3_module_map_combined.csv`，
> 本脚本不会覆盖它；per-direction 的两列表由脚本 4 从该审定文件派生。


In [ ]:
suppressPackageStartupMessages({library(dplyr); library(tidyr); library(readr); library(stringr); library(tibble)})

dir_io <- "/project/work/2026-09-01_GSEA_test/Data/"
conn <- read_csv(file.path(dir_io, "2026-09-02_2_fate_pathway_connection_all.csv"), show_col_types=FALSE) %>%
  filter(pass_U1)
cat("U1 唯一 pathway —— Macular:", n_distinct(conn$pathway[conn$direction=="Macular direction"]),
    "| Peripheral:", n_distinct(conn$pathway[conn$direction=="Peripheral direction"]), "\n")

In [ ]:
# ---- module 词典（有序：先匹配先生效；特异规则在前）--------------------------
dict <- tribble(
  ~regex, ~module,
  # 细胞骨架/纤毛（AXONEME 必须在 AXON 之前）
  "AXONEME|AXONEMAL|DYNEIN|CILIUM|MICROTUBULE|ACTIN|ORGANELLE_FISSION", "Cytoskeleton, cilium & organelle dynamics",
  # 轴突导向（RETINAL_GANGLION_CELL_AXON_GUIDANCE 必须在视网膜规则之前）
  "AXON|NEURON_PROJECTION_GUIDANCE|SEMAPHORIN|^GOBP_TAXIS$", "Neuronal development & axon guidance",
  # 免疫（免疫细胞凋亡归免疫，须在通用凋亡之前）
  "LEUKOCYTE|LYMPHOCYTE|LYMPHOID|T_CELL|B_CELL_RECEPTOR|MHC_CLASS|IMMUNE|ANTIMICROBIAL|HEMOPOIE", "Immune & inflammatory response",
  "APOPTOTIC|NEURON_DEATH|CELL_DEATH", "Cell death & apoptosis",
  # 光感受/感光转导
  "PHOTOTRANSDUCTION|DETECTION_OF_LIGHT|DETECTION_OF_VISIBLE|DETECTION_OF_ABIOTIC|DETECTION_OF_STIMULUS|SENSORY_PERCEPTION|RESPONSE_TO_LIGHT|RESPONSE_TO_RADIATION", "Phototransduction & light response",
  "PHOTORECEPTOR|CAMERA_TYPE_EYE|EYE_MORPHOGENESIS|RETINA|SENSORY_ORGAN|SENSORY_SYSTEM_DEVELOPMENT|EAR_DEVELOPMENT|MECHANORECEPTOR", "Eye & sensory organ development",
  "SYNAPSE|SYNAPTIC|NEUROTRANSMITTER|DENDRITIC_SPINE|GABA_SYNTHESIS|NEURONAL_SYSTEM|POSTSYNAP", "Synaptic signaling & neurotransmission",
  "GLIAL|GLIOGENESIS", "Glial development",
  "NEURON|NEURAL|NERVOUS_SYSTEM|FOREBRAIN|TELENCEPHALON|MIDBRAIN|SUBSTANTIA_NIGRA|SPINAL_CORD", "Neuronal development & axon guidance",
  "LEARNING|MEMORY|COGNITION|BEHAVIOR", "Cognition & behavior",
  # 信号转导（CYCLIC_NUCLEOTIDE 须在核苷酸代谢之前）
  "G_PROTEIN_COUPLED|GPCR|ADENYLATE_CYCLASE|G_ALPHA_I|CA2_PATHWAY|SECOND_MESSENGER|CYCLIC_NUCLEOTIDE", "GPCR & second-messenger signaling",
  "RHO_|RHOA|RAC1|RAC2|CDC42|GTPASE", "Small GTPase signaling",
  "WNT|PCP_|TCF_DEPENDENT|BETA_CATENIN", "WNT & PCP signaling",
  "MAPK|ERK1|ERBB|PI3K|MET_ACTIVATES|MET_ACTIVITY|NGF_|INSULIN|SERINE_THREONINE_KINASE|RESPONSE_TO_BMP|NUCLEAR_EVENTS_KINASE|REGULATION_OF_BINDING|CELL_SURFACE_RECEPTOR_SIGNALING", "Growth factor & kinase signaling",
  # 能量与核苷酸代谢（PROTON_TRANSMEMBRANE 归线粒体，须在通用 transport 之前）
  "OXIDATIVE_PHOSPHORYLATION|AEROBIC_RESPIRATION|CELLULAR_RESPIRATION|ATP_|TCA_CYCLE|CITRIC_ACID|RESPIRATORY_ELECTRON|CRISTAE|CHEMIOSMOTIC|PROTON_TRANSMEMBRANE|ENERGY_DERIVATION|PRECURSOR_METABOLITES_AND_ENERGY", "Mitochondrial energy metabolism",
  "NUCLEOSIDE|NUCLEOTIDE|NUCLEOBASE|RIBOSE_PHOSPHATE|PURINE", "Nucleotide metabolism",
  # 糖基化/分泌运输、蛋白降解、染色质
  "GLYCOSYLATION|GAG_SYNTHESIS|ER_TO_GOLGI|COPI|TRANSPORT_TO_THE_GOLGI|LYSOSOME|RECEPTOR_RECYCLING|REGULATION_OF_SECRETION", "Protein glycosylation & secretory trafficking",
  "PROTEOLYSIS|PEPTIDASE|PROTEIN_CATABOLIC|COMPONENT_DISASSEMBLY", "Protein catabolism & turnover",
  "CHROMATIN|NUCLEOSOME|SPLICING", "Chromatin & RNA processing",
  "CARBOHYDRATE|LIPID|ORGANIC_HYDROXY_COMPOUND_METABOLIC|ORGANIC_HYDROXY_COMPOUND_BIOSYNTHETIC|AMINO_ACIDS_AND_DERIVATIVES|POLYAMINE|VITAMIN|PEPTIDE_HORMONE_METABOLISM", "Small-molecule & lipid metabolism",
  # 转运（所有 *_TRANSPORT 统一归此，规则简单一致）
  "TRANSPORT|TRANSPORTER|ION_HOMEOSTASIS|ION_CHANNEL|IMPORT", "Ion & solute transport",
  "ADHESION|CELL_JUNCTION|JUNCTION_ORGANIZATION|EXTRACELLULAR_MATRIX|COLLAGEN|INTEGRIN|EXTERNAL_ENCAPSULATING|CELL_SUBSTRATE", "Cell adhesion, junction & ECM",
  "SENESCENCE|SASP|OXIDATIVE_STRESS|RESPONSE_TO_METAL_ION|CADMIUM|INORGANIC_SUBSTANCE|RESPONSE_TO_STRESS|RESPONSE_TO_EXTERNAL_STIMULUS|ORGANIC_CYCLIC_COMPOUND", "Senescence & stress response",
  "VASCUL|BLOOD_VESSEL|AORTA|ARTERY|CIRCULATORY|BLOOD_CIRCULATION|BLOOD_PRESSURE|TUBE_SIZE", "Vascular & circulatory system",
  "CARDIAC|HEART|MUSCLE|OUTFLOW_TRACT", "Cardiac & muscle function",
  # 器官发育（具名器官在通用增殖规则之前）
  "KIDNEY|GLOMERUL|METANEPHROS|MESONEPHROS|UROGENITAL|SKELETAL_SYSTEM|OSSIFICATION|OSTEOBLAST|BONE_|APPENDAGE|PANCREAS|ENDOCRINE_SYSTEM|REPRODUCTIVE|PROSTATE|EMBRYONIC|EMBRYO_DEVELOPMENT|GERM_LAYER|MESODERM|PATTERN_SPECIFICATION|AXIS_SPECIFICATION|SPECIFICATION_OF_SYMMETRY", "Organ development & morphogenesis",
  "PROLIFERATION|CELL_GROWTH|REGULATION_OF_GROWTH|DEVELOPMENTAL_GROWTH|CELL_DIFFERENTIATION|LOCOMOTION|MEIOTIC_CELL_CYCLE|MACROMOLECULE_BIOSYNTHETIC", "Cell growth, proliferation & differentiation",
  "EPITHELI|MESENCHYMAL|COLUMNAR_CUBOIDAL|MORPHOGENESIS", "Organ development & morphogenesis",
  "HOMEOSTASIS|GASEOUS_EXCHANGE|RESPIRATORY_SYSTEM|CIRCADIAN|MULTICELLULAR_ORGANISMAL|DIGESTIVE_SYSTEM", "Tissue & organismal homeostasis"
)
cat("词典规则数:", nrow(dict), "| module 数:", n_distinct(dict$module), "\n")

In [ ]:
# ---- 应用词典：返回首个匹配的 module ----------------------------------------
assign_module <- function(p) {
  hit <- which(str_detect(p, dict$regex))
  if (length(hit) == 0) NA_character_ else dict$module[min(hit)]
}
map_tbl <- tibble(pathway = sort(unique(conn$pathway))) %>%
  mutate(module = vapply(pathway, assign_module, character(1)))

cat("未归类 pathway 数:", sum(is.na(map_tbl$module)), "\n")
if (any(is.na(map_tbl$module))) print(map_tbl$pathway[is.na(map_tbl$module)])

In [ ]:
# ---- 输出两个两列表格（pathway, module）------------------------------------
for (d in c("Macular", "Peripheral")) {
  pw <- sort(unique(conn$pathway[conn$direction == paste0(d, " direction")]))
  out <- map_tbl %>% filter(pathway %in% pw) %>% arrange(module, pathway)
  write_csv(out, file.path(dir_io, paste0("2026-09-02_3_module_map_", d, "_draft.csv")), na="NA")
  cat(d, ": ", nrow(out), " pathway / ", n_distinct(out$module), " module\n", sep="")
}

# 合并长表（direction 列），便于跨方向比较
combined <- conn %>% distinct(direction, pathway) %>% left_join(map_tbl, by="pathway") %>%
  arrange(module, pathway, direction)
write_csv(combined, file.path(dir_io, "2026-09-02_3_module_map_combined_draft.csv"), na="NA")

# 共享 module 空间自检：同名 pathway 两方向归属必须一致
chk <- combined %>% group_by(pathway) %>% summarise(n_mod = n_distinct(module), .groups="drop")
cat("共享 module 空间自检 —— 归属不一致的 pathway 数:", sum(chk$n_mod > 1), "\n")

In [ ]:
# ---- module 规模概览（双方向对照，27-module 全空间；0 = 该方向未涉及）-------
ov <- conn %>% distinct(direction, pathway) %>% left_join(map_tbl, by="pathway") %>%
  count(module, direction) %>%
  pivot_wider(names_from=direction, values_from=n, values_fill=0) %>%
  rename(Macular=`Macular direction`, Peripheral=`Peripheral direction`) %>%
  mutate(shared = Macular>0 & Peripheral>0) %>%
  arrange(desc(Macular + Peripheral))
print(as.data.frame(ov))
cat("\nmodule 全空间:", nrow(ov), "| 两方向共有:", sum(ov$shared),
    "| 仅 Macular:", sum(ov$Peripheral==0), "| 仅 Peripheral:", sum(ov$Macular==0), "\n")
write_csv(ov, file.path(dir_io, "2026-09-02_3_module_size_overview_draft.csv"), na="NA")

In [ ]:
# ---- 参数与 sessionInfo ----------------------------------------------------
write_csv(dict, file.path(dir_io, "2026-09-02_3_module_dictionary.csv"))
writeLines(c("date_str: 2026-09-02_3  (pathway -> module 生物学功能划分)",
  "input: 2026-09-02_2_fate_pathway_connection_all.csv (pass_U1)",
  "method: 有序关键词词典, 先匹配先生效; 全 celltype/全方向共用同一份映射",
  "shared module space: 词典应用于双方向 pathway 并集, 再按方向拆表 -> 同名 pathway 归属必然一致",
  paste0("rules: ", nrow(dict), " | modules: ", n_distinct(dict$module)),
  "outputs: module_map_Macular.csv, module_map_Peripheral.csv (两列: pathway, module)",
  "         module_map_combined.csv (direction 长表), module_size_overview.csv, module_dictionary.csv",
  "", capture.output(sessionInfo())),
  file.path(dir_io, "2026-09-02_3_parameters_sessionInfo.txt"))
cat("完成。\n")